# 01 — Connect and Explore

This notebook verifies connectivity to all TableSchnappi services running on RHOAI.

> **Setup:** before running any cells, set `CLUSTER_DOMAIN` to your OpenShift apps
> domain in the configuration cell below (or export it as an environment variable).

| Service | Address |
|---------|----------|
| MLflow | `https://rh-ai.<your-cluster-apps-domain>/mlflow` |
| vLLM | `$MODEL_URL (see the configuration cell)` |
| Guardrails | `$GUARDRAILS_URL (see the configuration cell)` |
| MCP server | `$MCP_URL (see the configuration cell)` |

**Run every cell top-to-bottom.** Expected outcome: all services respond, MLflow shows
the `restaurant-chatbot-traces` experiment, and the MCP server lists 7 tools.

> This notebook is read-only — it makes no changes to prompts or data.
> Open `02_prompt_tuning.ipynb` for the hands-on exercise.

## Configuration

Everything this notebook needs is set in the next cell. In the workshop
workbench the values arrive as environment variables and you can run it as
is; anywhere else, replace the `EDIT ME` placeholders with your own
endpoint, model and namespace. The cell stops with a clear message if
anything is still unset.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit the values below for YOUR cluster.
#
# Inside the workshop workbench these arrive as environment variables and
# you can run this cell unchanged. Anywhere else — a laptop, a different
# namespace, a MaaS gateway — fill in the placeholders marked EDIT ME.
# ══════════════════════════════════════════════════════════════════════
import os

# Your project/namespace. The workshop chart installs into
# tableschnappi-<your-username>, not plain "tableschnappi".
NAMESPACE = os.environ.get("MLFLOW_WORKSPACE") or "EDIT ME: your-namespace"

# Where the model lives. No trailing /v1 — it is appended where needed.
#   in-cluster: http://<model>-predictor.<ns>.svc.cluster.local:8080
#   MaaS:       https://<maas-host>
MODEL_URL  = os.environ.get("VLLM_URL")   or "EDIT ME: https://your-model-endpoint"
MODEL_NAME = os.environ.get("VLLM_MODEL") or "EDIT ME: your-model-id"

# Only needed for an authenticated gateway such as MaaS.
MODEL_API_KEY = os.environ.get("MODEL_API_KEY", "EMPTY")

# Derived from NAMESPACE unless the workbench told us otherwise.
GUARDRAILS_URL = os.environ.get("GUARDRAILS_URL") or f"http://tableschnappi-guardrails.{NAMESPACE}.svc.cluster.local:80"
ASSISTANT_URL  = os.environ.get("ASSISTANT_URL")  or f"http://tableschnappi-mlflow-proxy.{NAMESPACE}.svc.cluster.local:8000"
MCP_URL        = os.environ.get("MCP_URL")        or f"http://tableschnappi-mcp.{NAMESPACE}.svc.cluster.local:3000/mcp"

MLFLOW_URI        = os.environ.get("MLFLOW_TRACKING_URI") or "EDIT ME: https://mlflow.redhat-ods-applications.svc:8443/mlflow"
MLFLOW_EXPERIMENT = "restaurant-chatbot-traces"

# MLflow on RHOAI authenticates with the pod's service-account token and
# serves a self-signed certificate.
os.environ["MLFLOW_TRACKING_URI"]          = MLFLOW_URI
os.environ["MLFLOW_TRACKING_AUTH"]         = "kubernetes-namespaced"
os.environ["MLFLOW_K8S_INTEGRATION"]       = "true"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
os.environ["OPENAI_API_KEY"]               = MODEL_API_KEY

_unset = [n for n, v in [
    ("NAMESPACE", NAMESPACE), ("MODEL_URL", MODEL_URL),
    ("MODEL_NAME", MODEL_NAME), ("MLFLOW_URI", MLFLOW_URI),
] if str(v).startswith("EDIT ME")]

for n, v in [("NAMESPACE", NAMESPACE), ("MODEL_URL", MODEL_URL), ("MODEL_NAME", MODEL_NAME),
             ("GUARDRAILS_URL", GUARDRAILS_URL), ("ASSISTANT_URL", ASSISTANT_URL),
             ("MLFLOW_URI", MLFLOW_URI)]:
    print(f"{n:16} {v}")

if _unset:
    raise SystemExit(
        "\nStill to fill in: " + ", ".join(_unset) +
        "\nEdit them at the top of this cell and run it again."
    )
print("\nConfiguration OK.")


In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "mlflow[kubernetes]>=3.11", "httpx"],
    check=True,
)
print("Dependencies ready.")

In [ ]:
import os, json
from urllib.parse import urlparse

def detect_cluster_domain():
    """Auto-detect the OpenShift apps domain."""
    if os.environ.get("CLUSTER_DOMAIN"):
        return os.environ["CLUSTER_DOMAIN"]
    # Parse from MLFLOW_TRACKING_URI injected by RHOAI workbench annotation
    mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if mlflow_uri:
        host = urlparse(mlflow_uri).hostname or ""
        if "." in host:
            return host.split(".", 1)[1]
    # Query OpenShift API for ingress domain (works inside pods)
    try:
        import urllib.request, ssl
        token = open("/var/run/secrets/kubernetes.io/serviceaccount/token").read()
        ctx = ssl.create_default_context(
            cafile="/var/run/secrets/kubernetes.io/serviceaccount/ca.crt"
        )
        req = urllib.request.Request(
            "https://kubernetes.default.svc/apis/config.openshift.io/v1/ingresses/cluster",
            headers={"Authorization": f"Bearer {token}"},
        )
        data = json.loads(urllib.request.urlopen(req, context=ctx, timeout=5).read())
        return data["spec"]["domain"]
    except Exception:
        pass
    return ""

CLUSTER_DOMAIN = detect_cluster_domain()
if not CLUSTER_DOMAIN:
    # The workbench injects CLUSTER_DOMAIN directly now. Off-cluster it is
    # only used to build convenience links, so warn rather than abort —
    # this used to be an assert, which stopped a laptop reader one cell
    # after the CONFIGURATION cell told them a laptop was fine.
    print("CLUSTER_DOMAIN not set — the links printed below will be "
          "incomplete. Set it to your apps domain: "
          "oc get ingresses.config/cluster -o jsonpath='{.spec.domain}'")

# MLFLOW_URI comes from the configuration cell
RHOAI_DASHBOARD  = f"https://rhods-dashboard.{CLUSTER_DOMAIN}"
TABLESCHNAPPI_UI = f"https://{NAMESPACE}-frontend-{NAMESPACE}.{CLUSTER_DOMAIN}"

print("CLUSTER_DOMAIN  :", CLUSTER_DOMAIN)
print("MLFLOW_URI      :", MLFLOW_URI)
print("RHOAI_DASHBOARD :", RHOAI_DASHBOARD)
print("TABLESCHNAPPI_UI:", TABLESCHNAPPI_UI)

In [ ]:
os.environ["MLFLOW_TRACKING_URI"]          = MLFLOW_URI
os.environ["MLFLOW_TRACKING_AUTH"]         = "kubernetes-namespaced"
os.environ["MLFLOW_K8S_INTEGRATION"]       = "true"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

print("MLFLOW_TRACKING_URI :", os.environ["MLFLOW_TRACKING_URI"])
print("MLFLOW_TRACKING_AUTH:", os.environ["MLFLOW_TRACKING_AUTH"])

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
client = mlflow.tracking.MlflowClient()

experiments = client.search_experiments()
print(f"Found {len(experiments)} experiment(s):\n")
for exp in experiments:
    marker = "  <-- target" if exp.name == "restaurant-chatbot-traces" else ""
    print(f"  [{exp.experiment_id:>3}] {exp.name}{marker}")

target = next(
    (e for e in experiments if e.name == "restaurant-chatbot-traces"), None
)
if target:
    print(f"\nrestaurant-chatbot-traces experiment ID: {target.experiment_id}")
else:
    print(
        "\nWarning: restaurant-chatbot-traces not found. "
        "Generate a trace first by chatting in the app."
    )

In [ ]:
# Load the production prompt from the MLflow Prompt Registry
try:
    prompt_obj = mlflow.genai.load_prompt("prompts:/restaurant-assistant-prompt@latest")
    print("=== restaurant-assistant-prompt@latest ===")
    print("-" * 60)
    print(prompt_obj.template)
    print("-" * 60)
except Exception as exc:
    print(f"Could not load prompt: {exc}")
    print("Hint: run seed-mlflow-prompts.py first to seed the registry.")

In [ ]:
import httpx

VLLM_URL = f"{MODEL_URL}/v1/chat/completions"

payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "You are a helpful restaurant assistant."},
        {"role": "user",   "content": "How many guests can typically sit at a standard restaurant table?"},
    ],
    "max_tokens": 80,
    "temperature": 0.1,
}

resp = httpx.post(VLLM_URL, json=payload, timeout=30, verify=False)
resp.raise_for_status()
data = resp.json()

print("Model   :", data["model"])
print("Response:", data["choices"][0]["message"]["content"])
print("Tokens  :", data["usage"]["total_tokens"])

In [ ]:
GUARDRAILS_CHAT_URL = f"{GUARDRAILS_URL}/v1/chat/completions"
MODEL = MODEL_NAME
BLOCK_PHRASES = [
    # Must match the chart's Colang refusal and app.py's
    # GUARDRAILS_REFUSAL_MARKER. Drifted once; every eval then reported 0%.
    "i'm here to help with restaurant reservations",
]

off_topic = "What is the capital of France?"

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": off_topic}],
    "max_tokens": 80,
}

resp = httpx.post(GUARDRAILS_CHAT_URL, json=payload, timeout=20, verify=False)
resp.raise_for_status()
content = resp.json()["choices"][0]["message"]["content"]
blocked = any(p in content.lower() for p in BLOCK_PHRASES)

print(f"Prompt  : {off_topic}")
print(f"Response: {content}")
print()
if blocked:
    print("Status  : BLOCKED -- guardrails are working correctly.")
else:
    print("Status  : NOT BLOCKED -- check the guardrails configuration.")

## All checks complete!

| Service | Status |
|---------|--------|
| MLflow | Connected — browse traces in the RHOAI Dashboard |
| vLLM | `$MODEL_NAME` responding |
| Guardrails | Blocking off-topic prompts correctly |
| MCP server | 7 tools listed |

**Links** (replace `<your-cluster-apps-domain>` with your `CLUSTER_DOMAIN` value)
- MLflow UI: `https://rh-ai.<your-cluster-apps-domain>/mlflow` (workspace: `tableschnappi`)
- RHOAI Dashboard: `https://rhods-dashboard.<your-cluster-apps-domain>`
- TableSchnappi app: `https://<release>-frontend-<namespace>.<your-cluster-apps-domain>`

**Next**: open `02_prompt_tuning.ipynb` to improve the restaurant assistant prompt.